# NB2 — Real Ablations + FSK Class + Train-High/Test-Low Generalization

Addresses:
- **R3 pt.8**: Real ablations (w/o GASF-CNN, w/o Attention Gate, w/o Label Smoothing) — all measured, single re-trained model each, against a re-trained Full Model reference
- **R3 pt.6**: "Lean" variant removing the redundant gated EMD path (original gate set a₁≈0)
- **R4 pt.6**: FSK added as a 5th modulation class, full 5-class model trained and evaluated incl. FSK-specific per-SNR accuracy
- **R3 pt.9**: Train-High/Test-Low generalization experiment (train on SNR∈{5,10,20}, evaluate on all SNRs)

Run on Kaggle T4 after NB1. Exports `nb2_export.pkl`.

In [ ]:
!pip install EMD-signal

In [ ]:
import os
OUTPUT_DIR = '/kaggle/working/paper_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Figures will be saved to: {os.path.abspath(OUTPUT_DIR)}')

# -*- coding: utf-8 -*-
"""
NB2 — Real Ablations + FSK Class + Train-High/Test-Low Generalization
Addresses (this notebook):
  - R3 pt.8: real ablations — w/o GASF-CNN path, w/o Attention Gate,
    w/o Label Smoothing — measured, not estimated
  - R3 pt.6: remove redundant gated EMD path (a1=0 in original gate;
    EMD already reaches fusion via skip connection) -> "Lean" variant
  - R4 pt.6: add FSK as a 5th modulation class
  - R3 pt.9: train only on high SNR (>=5dB), test on low SNR (<=0dB)
    to test whether modulation-specific (not SNR-specific) features
    are learned
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model, mixed_precision
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical
from scipy.signal import hilbert, butter, lfilter
from scipy.stats import skew, kurtosis, entropy
from PyEMD import EMD
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time, gc, logging, warnings, pickle
logging.getLogger('PyEMD').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

mixed_precision.set_global_policy('mixed_float16')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU ready: {len(gpus)} device(s), mixed_float16 enabled")

# ══════════════════════════════════════════════════════════════════════════
# SCALE PARAMETERS  (same as NB1, for the core re-trained model)
# ══════════════════════════════════════════════════════════════════════════
N_D         = 1500
GASF_D      = 96
SEQ_D       = 256
BATCH_D     = 64
EPOCHS_D    = 80
PATIENCE_D  = 20
N_ENSEMBLE  = 5

SNRs_to_test = [-10, -5, 0, 5, 10, 20]
CLASSES_4 = ['AM', 'PM', 'FM', 'SSB']
CLASSES_5 = ['AM', 'PM', 'FM', 'SSB', 'FSK']

# Ablation re-trains: 1 model each (not 5-ensemble) to fit time budget;
# Full Model also re-trained as 1 model for apples-to-apples ablation deltas.
ABLATION_EPOCHS   = 80
ABLATION_PATIENCE = 20

# FSK extension dataset (smaller scale to fit budget)
N_FSK       = 1000   # per class per SNR for the 5-class experiment
GASF_FSK    = 96
SEQ_FSK     = 256

# Train-high/test-low experiment uses the same N_D AWGN 4-class dataset
HIGH_SNRS = [5, 10, 20]
LOW_SNRS  = [-10, -5, 0]

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'figure.dpi': 120, 'savefig.dpi': 200,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.15,
})

def butter_lowpass(cutoff, fs, order=9):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low', analog=False)
    return b, a

print("NB2 setup complete.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 1. DATA GENERATION
# (a) 4-class AWGN dataset (same generator as original/NB1) for ablations
#     and the train-high/test-low experiment
# (b) 5-class dataset adding FSK (R4 pt.6)
# ══════════════════════════════════════════════════════════════════════════
def generate_signals_4class(n_per_class, snrs=SNRs_to_test, num_samples=1024):
    total = n_per_class * 4 * len(snrs)
    print(f"Generating {total} 4-class signals ({n_per_class}/class/SNR)...")
    fs, fc = 100000, 5000
    t = np.arange(num_samples) / fs
    X, Y, SNR_labels = [], [], []
    for snr in snrs:
        for _ in range(n_per_class):
            noise    = np.random.randn(num_samples)
            b, a     = butter_lowpass(1000, fs, order=9)
            m_t      = lfilter(b, a, noise)
            m_t_norm = m_t / np.max(np.abs(m_t))
            am   = (1 + 0.8*m_t_norm)*np.cos(2*np.pi*fc*t)
            pm   = np.cos(2*np.pi*fc*t + np.pi*m_t_norm)
            fm   = np.cos(2*np.pi*fc*t + 2*np.pi*1000*np.cumsum(m_t)/fs)
            m_hat = np.imag(hilbert(m_t))
            ssb  = m_t*np.cos(2*np.pi*fc*t) - m_hat*np.sin(2*np.pi*fc*t)
            for idx, sig in enumerate([am, pm, fm, ssb]):
                sp  = np.mean(sig**2)
                np_ = sp / (10**(snr/10))
                ns  = sig + np.sqrt(np_)*np.random.randn(num_samples)
                X.append((ns - ns.mean()) / ns.std())
                Y.append(idx)
                SNR_labels.append(snr)
    return (np.array(X, dtype=np.float32),
            np.array(Y, dtype=np.int8),
            np.array(SNR_labels, dtype=np.int8))

def generate_signals_5class(n_per_class, snrs=SNRs_to_test, num_samples=1024):
    """4 original classes + FSK (binary FSK, two tones around fc)."""
    total = n_per_class * 5 * len(snrs)
    print(f"Generating {total} 5-class (incl. FSK) signals ({n_per_class}/class/SNR)...")
    fs, fc = 100000, 5000
    fdev = 1500  # frequency deviation for FSK tones
    t = np.arange(num_samples) / fs
    X, Y, SNR_labels = [], [], []
    for snr in snrs:
        for _ in range(n_per_class):
            noise    = np.random.randn(num_samples)
            b, a     = butter_lowpass(1000, fs, order=9)
            m_t      = lfilter(b, a, noise)
            m_t_norm = m_t / np.max(np.abs(m_t))
            am   = (1 + 0.8*m_t_norm)*np.cos(2*np.pi*fc*t)
            pm   = np.cos(2*np.pi*fc*t + np.pi*m_t_norm)
            fm   = np.cos(2*np.pi*fc*t + 2*np.pi*1000*np.cumsum(m_t)/fs)
            m_hat = np.imag(hilbert(m_t))
            ssb  = m_t*np.cos(2*np.pi*fc*t) - m_hat*np.sin(2*np.pi*fc*t)
            # FSK: random binary symbol stream switching between fc-fdev and fc+fdev
            n_symbols = 16
            symbol_len = num_samples // n_symbols
            bits = np.random.randint(0, 2, n_symbols)
            freqs = np.where(bits == 0, fc - fdev, fc + fdev)
            fsk = np.zeros(num_samples)
            phase = 0.0
            for si, f in enumerate(freqs):
                seg = np.arange(symbol_len)
                fsk_seg = np.cos(2*np.pi*f*seg/fs + phase)
                start = si*symbol_len
                end = start + symbol_len
                fsk[start:end] = fsk_seg[:end-start]
                phase += 2*np.pi*f*symbol_len/fs
            if num_samples % n_symbols != 0:
                fsk[end:] = fsk[:num_samples-end]
            for idx, sig in enumerate([am, pm, fm, ssb, fsk]):
                sp  = np.mean(sig**2)
                np_ = sp / (10**(snr/10))
                ns  = sig + np.sqrt(np_)*np.random.randn(num_samples)
                X.append((ns - ns.mean()) / ns.std())
                Y.append(idx)
                SNR_labels.append(snr)
    return (np.array(X, dtype=np.float32),
            np.array(Y, dtype=np.int8),
            np.array(SNR_labels, dtype=np.int8))

# ── 4-class AWGN (for ablations + train-high/test-low) ────────────────────
X_raw, Y, SNR_labels = generate_signals_4class(N_D)

# ── 5-class AWGN incl. FSK (smaller scale) ─────────────────────────────────
X_raw_5, Y_5, SNR_labels_5 = generate_signals_5class(N_FSK)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 2. FEATURE EXTRACTION (shared functions)
# ══════════════════════════════════════════════════════════════════════════
def extract_emd_features(signal):
    emd  = EMD()
    imfs = emd.emd(signal, max_imf=3)
    features = []
    for i in range(3):
        if i < len(imfs):
            imf   = imfs[i]
            power = np.abs(imf)**2
            prob  = power / (np.sum(power) + 1e-10)
            features.extend([np.std(imf), skew(imf), kurtosis(imf),
                              np.sum(power), entropy(prob)])
        else:
            features.extend([0.0] * 5)
    return np.array(features, dtype=np.float32)

def compute_gasf(signal, size):
    sr     = np.interp(np.linspace(0, len(signal), size),
                       np.arange(len(signal)), signal)
    lo, hi = sr.min(), sr.max()
    n      = (sr - lo) / (hi - lo + 1e-8) * 2 - 1
    phi    = np.arccos(np.clip(n, -1, 1))
    c, s   = np.cos(phi), np.sin(phi)
    return (np.outer(c, c) - np.outer(s, s)).astype(np.float32)

def extract_sequence(signal, seq_len):
    stride = max(1, len(signal) // seq_len)
    seq    = signal[::stride][:seq_len]
    if len(seq) < seq_len:
        seq = np.pad(seq, (0, seq_len - len(seq)))
    return seq.astype(np.float32).reshape(seq_len, 1)

def extract_all(X_raw_arr, gasf_size, seq_len, tag=""):
    emd_l, gaf_l, seq_l = [], [], []
    for i, sig in enumerate(X_raw_arr):
        if i % 5000 == 0 and i > 0:
            print(f"  [{tag}] {i}/{len(X_raw_arr)} processed...")
        emd_l.append(extract_emd_features(sig))
        gaf_l.append(compute_gasf(sig, gasf_size))
        seq_l.append(extract_sequence(sig, seq_len))
    X_emd_ = np.array(emd_l, dtype=np.float32).reshape(-1, 15, 1)
    X_gaf_ = np.array(gaf_l, dtype=np.float32).reshape(-1, gasf_size, gasf_size, 1)
    X_seq_ = np.array(seq_l, dtype=np.float32)
    return X_emd_, X_gaf_, X_seq_

# ── 4-class AWGN dataset features ──────────────────────────────────────────
print(f"\nExtracting features for {len(X_raw)} 4-class AWGN signals...")
X_emd, X_gaf_d, X_seq_d = extract_all(X_raw, GASF_D, SEQ_D, tag="4-class")
del X_raw; gc.collect()

Y_onehot = to_categorical(Y, num_classes=4)
train_idx, test_idx = train_test_split(
    np.arange(len(Y)), test_size=0.2, random_state=42, stratify=Y)

train_d  = {"emd_input": X_emd[train_idx], "gaf_input": X_gaf_d[train_idx], "seq_input": X_seq_d[train_idx]}
test_d   = {"emd_input": X_emd[test_idx],  "gaf_input": X_gaf_d[test_idx],  "seq_input": X_seq_d[test_idx]}
y_train_d  = Y_onehot[train_idx]
y_test     = Y_onehot[test_idx]
snr_test   = SNR_labels[test_idx]
snr_train  = SNR_labels[train_idx]
print(f"4-class data ready. Train: {len(train_idx)} | Test: {len(test_idx)}")

# ── Train-high/test-low split (R3 pt.9) ────────────────────────────────────
# Train ONLY on high-SNR samples (>=5dB) from the training pool, test on
# the FULL test set (all SNRs) to compare against the original split, AND
# specifically on the low-SNR subset.
high_mask_train = np.isin(snr_train, HIGH_SNRS)
hl_train_idx_local = np.where(high_mask_train)[0]
train_hl = {"emd_input": train_d["emd_input"][hl_train_idx_local],
             "gaf_input": train_d["gaf_input"][hl_train_idx_local],
             "seq_input": train_d["seq_input"][hl_train_idx_local]}
y_train_hl = y_train_d[hl_train_idx_local]
print(f"Train-high subset: {len(hl_train_idx_local)} samples "
      f"(SNRs {HIGH_SNRS}), full test set unchanged for evaluation.")

del X_emd, X_gaf_d, X_seq_d, Y, Y_onehot
gc.collect()

# ── 5-class (incl. FSK) dataset features ───────────────────────────────────
print(f"\nExtracting features for {len(X_raw_5)} 5-class (incl. FSK) signals...")
X_emd5, X_gaf5, X_seq5 = extract_all(X_raw_5, GASF_FSK, SEQ_FSK, tag="5-class")
del X_raw_5; gc.collect()

Y5_onehot = to_categorical(Y_5, num_classes=5)
train_idx5, test_idx5 = train_test_split(
    np.arange(len(Y_5)), test_size=0.2, random_state=42, stratify=Y_5)

train_5  = {"emd_input": X_emd5[train_idx5], "gaf_input": X_gaf5[train_idx5], "seq_input": X_seq5[train_idx5]}
test_5   = {"emd_input": X_emd5[test_idx5],  "gaf_input": X_gaf5[test_idx5],  "seq_input": X_seq5[test_idx5]}
y_train_5 = Y5_onehot[train_idx5]
y_test_5  = Y5_onehot[test_idx5]
snr_test_5 = SNR_labels_5[test_idx5]
print(f"5-class data ready. Train: {len(train_idx5)} | Test: {len(test_idx5)}")

del X_emd5, X_gaf5, X_seq5, Y_5, Y5_onehot
gc.collect()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 3. SHARED BLOCKS + ABLATION ARCHITECTURE VARIANTS
# Each ablation variant is a structural modification of build_triple_path,
# so the only difference between rows IS the named component (R3 pt.8).
# ══════════════════════════════════════════════════════════════════════════
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(max(filters//ratio, 1), activation='relu',
                      kernel_initializer='he_normal', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid',
                      kernel_initializer='he_normal', use_bias=False)(se)
    return layers.Multiply()([x, se])

class CosineAnnealingLR(tf.keras.callbacks.Callback):
    def __init__(self, initial_lr=0.0005, min_lr=1e-6, total_epochs=80):
        super().__init__()
        self.initial_lr = initial_lr
        self.min_lr     = min_lr
        self.total_epochs = total_epochs
    def on_epoch_begin(self, epoch, logs=None):
        decay  = 0.5*(1 + np.cos(np.pi*epoch/self.total_epochs))
        new_lr = self.min_lr + (self.initial_lr - self.min_lr)*decay
        self.model.optimizer.learning_rate.assign(float(new_lr))

def get_callbacks(epochs, patience):
    return [
        CosineAnnealingLR(0.0005, 1e-6, epochs),
        ReduceLROnPlateau('val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=0),
        EarlyStopping('val_loss', patience=patience, restore_best_weights=True)
    ]

def gaf_cnn_branch(gaf_input, gasf_size, dual_kernel=True, full_res=True):
    if dual_kernel:
        x = layers.Concatenate()([
            layers.Conv2D(16,(3,3),padding='same',activation='relu')(gaf_input),
            layers.Conv2D(16,(5,5),padding='same',activation='relu')(gaf_input),
        ])
    else:
        x = layers.Conv2D(32,(3,3),padding='same',activation='relu')(gaf_input)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = se_block(x)
    x = layers.Conv2D(64,(3,3),padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = se_block(x)
    n_filters = 128 if full_res else 64
    x = layers.Conv2D(n_filters,(3,3),padding='same',activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = se_block(x)
    return layers.Concatenate()([layers.GlobalAveragePooling2D()(x),
                                  layers.GlobalMaxPooling2D()(x)])

def build_emd_path(emd_input):
    emd_raw = layers.Flatten()(emd_input)
    x1 = layers.Dense(64, activation='relu')(emd_raw)
    x1 = layers.BatchNormalization()(x1)
    x1 = layers.Dense(32, activation='relu')(x1)
    return x1, emd_raw

def build_seq_path(seq_input):
    x3 = layers.Conv1D(32, 7, strides=2, padding='same', activation='relu')(seq_input)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Conv1D(64, 5, strides=2, padding='same', activation='relu')(x3)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Conv1D(64, 3, strides=1, padding='same', activation='relu')(x3)
    x3 = layers.BatchNormalization()(x3)
    x3 = layers.Bidirectional(layers.GRU(64, return_sequences=True, dropout=0.1))(x3)
    x3 = layers.Bidirectional(layers.GRU(32, return_sequences=False, dropout=0.1))(x3)
    return x3

def fusion_head(merged, n_classes=4, label_smoothing=0.05):
    z = layers.Dense(256, activation='relu')(merged)
    z = layers.BatchNormalization()(z)
    z = layers.Dropout(0.25)(z)
    z = layers.Dense(128, activation='relu')(z)
    z = layers.Dropout(0.15)(z)
    z = layers.Dense(64,  activation='relu')(z)
    out = layers.Dense(n_classes, activation='softmax', dtype='float32')(z)
    return out

def compile_model(model, label_smoothing=0.05, lr=0.0003):
    if label_smoothing > 0:
        loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)
    else:
        loss = 'categorical_crossentropy'
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                   loss=loss, metrics=['accuracy'])
    return model

# ══════════════════════════════════════════════════════════════════════════
# FULL MODEL (Proposed) — re-trained here as single model for fair ablation
# baseline comparison (each ablation also trains 1 model)
# ══════════════════════════════════════════════════════════════════════════
def build_full_model(n_classes=4, label_smoothing=0.05, gasf_size=GASF_D, seq_len=SEQ_D):
    emd_input = layers.Input(shape=(15,1), name="emd_input")
    gaf_input = layers.Input(shape=(gasf_size, gasf_size, 1), name="gaf_input")
    seq_input = layers.Input(shape=(seq_len, 1), name="seq_input")

    x1, emd_raw = build_emd_path(emd_input)
    x2 = gaf_cnn_branch(gaf_input, gasf_size, dual_kernel=True, full_res=(gasf_size==96))
    x3 = build_seq_path(seq_input)

    concat_all = layers.Concatenate()([x1, x2, x3, emd_raw])
    gate_hidden = layers.Dense(128, activation='relu')(concat_all)
    gate = layers.Dense(3, activation='softmax', name='path_gate')(gate_hidden)

    x1_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,0],1), name='gate_emd')([x1, gate])
    x2_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,1],1), name='gate_gaf')([x2, gate])
    x3_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,2],1), name='gate_gru')([x3, gate])

    merged = layers.Concatenate()([x1_g, x2_g, x3_g, emd_raw])
    out = fusion_head(merged, n_classes, label_smoothing)
    model = Model(inputs=[emd_input, gaf_input, seq_input], outputs=out, name="Full_Model")
    return compile_model(model, label_smoothing)

# ══════════════════════════════════════════════════════════════════════════
# ABLATION A: w/o GASF-CNN path (R3 pt.8)
# Remove x2 entirely; gate becomes 2-way (EMD, BiGRU)
# ══════════════════════════════════════════════════════════════════════════
def build_no_gasf():
    emd_input = layers.Input(shape=(15,1), name="emd_input")
    gaf_input = layers.Input(shape=(GASF_D, GASF_D, 1), name="gaf_input")  # unused, kept for API parity
    seq_input = layers.Input(shape=(SEQ_D, 1), name="seq_input")

    x1, emd_raw = build_emd_path(emd_input)
    x3 = build_seq_path(seq_input)

    concat_all = layers.Concatenate()([x1, x3, emd_raw])
    gate_hidden = layers.Dense(128, activation='relu')(concat_all)
    gate = layers.Dense(2, activation='softmax', name='path_gate')(gate_hidden)

    x1_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,0],1), name='gate_emd')([x1, gate])
    x3_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,1],1), name='gate_gru')([x3, gate])

    merged = layers.Concatenate()([x1_g, x3_g, emd_raw])
    out = fusion_head(merged, 4, 0.05)
    # gaf_input included as an input (ignored) so the same data dict can be passed
    dummy = layers.Lambda(lambda g: g*0.0)(layers.GlobalAveragePooling2D()(gaf_input))
    out = layers.Add()([out, layers.Dense(4, activation=None, dtype='float32',
                                            kernel_initializer='zeros', use_bias=False)(dummy)])
    out = layers.Activation('softmax', dtype='float32')(out)
    model = Model(inputs=[emd_input, gaf_input, seq_input], outputs=out, name="No_GASF_CNN")
    return compile_model(model, 0.05)

# ══════════════════════════════════════════════════════════════════════════
# ABLATION B: w/o Attention Gate (R3 pt.8)
# Replace gated weighting with simple concatenation (no gate layer at all)
# ══════════════════════════════════════════════════════════════════════════
def build_no_gate():
    emd_input = layers.Input(shape=(15,1), name="emd_input")
    gaf_input = layers.Input(shape=(GASF_D, GASF_D, 1), name="gaf_input")
    seq_input = layers.Input(shape=(SEQ_D, 1), name="seq_input")

    x1, emd_raw = build_emd_path(emd_input)
    x2 = gaf_cnn_branch(gaf_input, GASF_D, dual_kernel=True, full_res=True)
    x3 = build_seq_path(seq_input)

    # No gating — direct concatenation of all paths
    merged = layers.Concatenate()([x1, x2, x3, emd_raw])
    out = fusion_head(merged, 4, 0.05)
    model = Model(inputs=[emd_input, gaf_input, seq_input], outputs=out, name="No_Attention_Gate")
    return compile_model(model, 0.05)

# ══════════════════════════════════════════════════════════════════════════
# ABLATION C: w/o Label Smoothing (R3 pt.8)
# Identical to full model but label_smoothing=0
# ══════════════════════════════════════════════════════════════════════════
def build_no_label_smoothing():
    m = build_full_model(n_classes=4, label_smoothing=0.0)
    m._name = "No_Label_Smoothing"
    return m

# ══════════════════════════════════════════════════════════════════════════
# "LEAN" VARIANT: remove redundant gated EMD path (R3 pt.6)
# Original gate assigned a1=0 to gated EMD -> that path contributed nothing
# beyond the EMD skip connection (emd_raw). This variant removes x1/x1_g
# from the gate and fusion entirely, keeping only emd_raw as the EMD
# contribution, with a 2-way gate over (GASF, BiGRU).
# ══════════════════════════════════════════════════════════════════════════
def build_lean_model():
    emd_input = layers.Input(shape=(15,1), name="emd_input")
    gaf_input = layers.Input(shape=(GASF_D, GASF_D, 1), name="gaf_input")
    seq_input = layers.Input(shape=(SEQ_D, 1), name="seq_input")

    emd_raw = layers.Flatten()(emd_input)
    x2 = gaf_cnn_branch(gaf_input, GASF_D, dual_kernel=True, full_res=True)
    x3 = build_seq_path(seq_input)

    concat_all = layers.Concatenate()([x2, x3, emd_raw])
    gate_hidden = layers.Dense(128, activation='relu')(concat_all)
    gate = layers.Dense(2, activation='softmax', name='path_gate')(gate_hidden)

    x2_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,0],1), name='gate_gaf')([x2, gate])
    x3_g = layers.Lambda(lambda inp: inp[0]*tf.expand_dims(inp[1][:,1],1), name='gate_gru')([x3, gate])

    merged = layers.Concatenate()([x2_g, x3_g, emd_raw])
    out = fusion_head(merged, 4, 0.05)
    model = Model(inputs=[emd_input, gaf_input, seq_input], outputs=out, name="Lean_NoRedundantEMD")
    return compile_model(model, 0.05)

# ══════════════════════════════════════════════════════════════════════════
# 5-CLASS MODEL (incl. FSK)  (R4 pt.6)
# ══════════════════════════════════════════════════════════════════════════
def build_full_model_5class():
    return build_full_model(n_classes=5, label_smoothing=0.05,
                              gasf_size=GASF_FSK, seq_len=SEQ_FSK)

print("NB2 architectures defined: Full Model, No-GASF-CNN, No-Attention-Gate, "
      "No-Label-Smoothing, Lean (no redundant EMD), 5-class (+FSK)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 4. TRAINING — Real Ablations, Lean Variant, FSK 5-class, Train-High/Test-Low
# ══════════════════════════════════════════════════════════════════════════
def eval_per_snr_single(model, td, yt, snr_labels, snrs=SNRs_to_test):
    accs = []
    for snr in snrs:
        idx  = np.where(snr_labels == snr)[0]
        if len(idx) == 0:
            accs.append(np.nan); continue
        data = {k: v[idx] for k, v in td.items()}
        pr   = model.predict(data, verbose=0)
        acc  = np.mean(np.argmax(pr,1) == np.argmax(yt[idx],1)) * 100
        accs.append(acc)
    return accs

def print_header(name, desc):
    print("\n" + "█"*65)
    print(f"█  {name}")
    print(f"█  {desc}")
    print("█"*65)

def print_done(h):
    ep  = len(h.history['loss'])
    acc = h.history['val_accuracy'][-1]*100
    best_acc = max(h.history['val_accuracy'])*100
    print(f"  └─ Epochs: {ep} | Final val acc: {acc:.1f}% | Best val acc: {best_acc:.1f}%")

def print_results(name, accs, snrs=SNRs_to_test):
    print(f"\n  ╔══ {name} ══╗")
    for snr, acc in zip(snrs, accs):
        if np.isnan(acc):
            print(f"  ║  {snr:>4} dB :    N/A")
            continue
        bar = '█' * int(acc/5)
        print(f"  ║  {snr:>4} dB : {acc:>6.2f}%  {bar}")
    valid = [a for a in accs if not np.isnan(a)]
    print(f"  ║  Mean: {np.mean(valid):.2f}%")
    print(f"  ╚{'═'*42}╝")

ablation_results = {}
ablation_params  = {}

# ── FULL MODEL (re-trained as single model, for fair ablation deltas) ─────
print_header("FULL MODEL (re-trained, single)",
             "Reference for ablation deltas | All components active")
t0 = time.time()
tf.random.set_seed(0); np.random.seed(0)
m_full = build_full_model()
h = m_full.fit(train_d, y_train_d, validation_data=(test_d, y_test),
               batch_size=BATCH_D, epochs=ABLATION_EPOCHS,
               callbacks=get_callbacks(ABLATION_EPOCHS, ABLATION_PATIENCE), verbose=1)
print_done(h)
ablation_results['Full Model'] = eval_per_snr_single(m_full, test_d, y_test, snr_test)
ablation_params['Full Model']  = m_full.count_params()
print_results('Full Model', ablation_results['Full Model'])
print(f"  ⏱  {(time.time()-t0)/60:.1f} min")
m_full.save_weights('/kaggle/working/model_full_ablation_ref.weights.h5')
tf.keras.backend.clear_session(); gc.collect()

# ── ABLATION A: w/o GASF-CNN ────────────────────────────────────────────
print_header("ABLATION A — w/o GASF-CNN path", "R3 pt.8: GASF-CNN path removed entirely")
t0 = time.time()
tf.random.set_seed(0); np.random.seed(0)
m_nogasf = build_no_gasf()
h = m_nogasf.fit(train_d, y_train_d, validation_data=(test_d, y_test),
                 batch_size=BATCH_D, epochs=ABLATION_EPOCHS,
                 callbacks=get_callbacks(ABLATION_EPOCHS, ABLATION_PATIENCE), verbose=1)
print_done(h)
ablation_results['w/o GASF-CNN'] = eval_per_snr_single(m_nogasf, test_d, y_test, snr_test)
ablation_params['w/o GASF-CNN']  = m_nogasf.count_params()
print_results('w/o GASF-CNN', ablation_results['w/o GASF-CNN'])
print(f"  ⏱  {(time.time()-t0)/60:.1f} min")
tf.keras.backend.clear_session(); gc.collect()

# ── ABLATION B: w/o Attention Gate ──────────────────────────────────────
print_header("ABLATION B — w/o Attention Gate", "R3 pt.8: gate replaced by plain concatenation")
t0 = time.time()
tf.random.set_seed(0); np.random.seed(0)
m_nogate = build_no_gate()
h = m_nogate.fit(train_d, y_train_d, validation_data=(test_d, y_test),
                 batch_size=BATCH_D, epochs=ABLATION_EPOCHS,
                 callbacks=get_callbacks(ABLATION_EPOCHS, ABLATION_PATIENCE), verbose=1)
print_done(h)
ablation_results['w/o Attention Gate'] = eval_per_snr_single(m_nogate, test_d, y_test, snr_test)
ablation_params['w/o Attention Gate']  = m_nogate.count_params()
print_results('w/o Attention Gate', ablation_results['w/o Attention Gate'])
print(f"  ⏱  {(time.time()-t0)/60:.1f} min")
tf.keras.backend.clear_session(); gc.collect()

# ── ABLATION C: w/o Label Smoothing ─────────────────────────────────────
print_header("ABLATION C — w/o Label Smoothing", "R3 pt.8: label_smoothing=0 (identical architecture otherwise)")
t0 = time.time()
tf.random.set_seed(0); np.random.seed(0)
m_nols = build_no_label_smoothing()
h = m_nols.fit(train_d, y_train_d, validation_data=(test_d, y_test),
               batch_size=BATCH_D, epochs=ABLATION_EPOCHS,
               callbacks=get_callbacks(ABLATION_EPOCHS, ABLATION_PATIENCE), verbose=1)
print_done(h)
ablation_results['w/o Label Smoothing'] = eval_per_snr_single(m_nols, test_d, y_test, snr_test)
ablation_params['w/o Label Smoothing']  = m_nols.count_params()
print_results('w/o Label Smoothing', ablation_results['w/o Label Smoothing'])
print(f"  ⏱  {(time.time()-t0)/60:.1f} min")
tf.keras.backend.clear_session(); gc.collect()

# ── LEAN VARIANT: w/o redundant gated EMD path ──────────────────────────
print_header("LEAN VARIANT — w/o redundant gated EMD path",
             "R3 pt.6: original gate set a1≈0 (EMD path contributed nothing "
             "beyond skip connection); this variant removes it structurally")
t0 = time.time()
tf.random.set_seed(0); np.random.seed(0)
m_lean = build_lean_model()
h = m_lean.fit(train_d, y_train_d, validation_data=(test_d, y_test),
               batch_size=BATCH_D, epochs=ABLATION_EPOCHS,
               callbacks=get_callbacks(ABLATION_EPOCHS, ABLATION_PATIENCE), verbose=1)
print_done(h)
ablation_results['Lean (no redundant EMD)'] = eval_per_snr_single(m_lean, test_d, y_test, snr_test)
ablation_params['Lean (no redundant EMD)']  = m_lean.count_params()
print_results('Lean (no redundant EMD)', ablation_results['Lean (no redundant EMD)'])
print(f"  ⏱  {(time.time()-t0)/60:.1f} min")
tf.keras.backend.clear_session(); gc.collect()

# ══════════════════════════════════════════════════════════════════════════
# ABLATION SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═"*85)
print(f"{'VARIANT':<28}" + "".join([f"{s:>7}dB" for s in SNRs_to_test]) + f"{'Mean':>8}{'Δ vs Full':>11}{'Params':>12}")
print("─"*85)
full_mean = np.mean(ablation_results['Full Model'])
for name in ['Full Model', 'w/o GASF-CNN', 'w/o Attention Gate',
              'w/o Label Smoothing', 'Lean (no redundant EMD)']:
    accs = ablation_results[name]
    mean = np.mean(accs)
    delta = mean - full_mean
    print(f"{name:<28}" + "".join([f"{a:>9.2f}" for a in accs]) +
          f"{mean:>8.2f}" + f"{delta:>+11.2f}" + f"{ablation_params[name]:>12,}")
print("═"*85)

# ══════════════════════════════════════════════════════════════════════════
# FSK 5-CLASS MODEL (R4 pt.6)
# ══════════════════════════════════════════════════════════════════════════
print_header("5-CLASS MODEL (incl. FSK)", f"R4 pt.6 | {GASF_FSK}x{GASF_FSK} GASF | {SEQ_FSK}-step seq")
t0 = time.time()
tf.random.set_seed(0); np.random.seed(0)
m_5class = build_full_model_5class()
h = m_5class.fit(train_5, y_train_5, validation_data=(test_5, y_test_5),
                 batch_size=BATCH_D, epochs=ABLATION_EPOCHS,
                 callbacks=get_callbacks(ABLATION_EPOCHS, ABLATION_PATIENCE), verbose=1)
print_done(h)
fsk_accs = eval_per_snr_single(m_5class, test_5, y_test_5, snr_test_5)
print_results('5-Class (incl. FSK)', fsk_accs)

# Per-class breakdown including FSK specifically
pr_5 = m_5class.predict(test_5, verbose=0)
pred_5 = np.argmax(pr_5, 1)
true_5 = np.argmax(y_test_5, 1)
print("\nPer-class accuracy (5-class model, all SNRs):")
for ci, cname in enumerate(CLASSES_5):
    idx = np.where(true_5 == ci)[0]
    acc = np.mean(pred_5[idx] == true_5[idx]) * 100
    print(f"  {cname:<6}: {acc:.2f}%  (n={len(idx)})")

# FSK accuracy specifically per SNR
fsk_idx_all = np.where(true_5 == 4)[0]
print("\nFSK class accuracy per SNR:")
fsk_per_snr = []
for snr in SNRs_to_test:
    idx = fsk_idx_all[snr_test_5[fsk_idx_all] == snr]
    if len(idx) == 0:
        fsk_per_snr.append(np.nan); continue
    acc = np.mean(pred_5[idx] == 4) * 100
    fsk_per_snr.append(acc)
    print(f"  {snr:>4} dB: {acc:.2f}%  (n={len(idx)})")

print(f"\n  ⏱  {(time.time()-t0)/60:.1f} min | Params: {m_5class.count_params():,}")
m_5class.save_weights('/kaggle/working/model_5class_fsk.weights.h5')
tf.keras.backend.clear_session(); gc.collect()

# ══════════════════════════════════════════════════════════════════════════
# TRAIN-HIGH / TEST-LOW GENERALIZATION (R3 pt.9)
# Train only on SNR in {5,10,20}, test on FULL test set (all SNRs) —
# compare per-SNR accuracy against the original (full-range-trained) model.
# ══════════════════════════════════════════════════════════════════════════
print_header("TRAIN-HIGH / TEST-LOW GENERALIZATION", f"R3 pt.9 | Train SNRs={HIGH_SNRS}, evaluate on all SNRs")
t0 = time.time()
tf.random.set_seed(0); np.random.seed(0)
m_hl = build_full_model()
h = m_hl.fit(train_hl, y_train_hl, validation_data=(test_d, y_test),
              batch_size=BATCH_D, epochs=ABLATION_EPOCHS,
              callbacks=get_callbacks(ABLATION_EPOCHS, ABLATION_PATIENCE), verbose=1)
print_done(h)
hl_accs = eval_per_snr_single(m_hl, test_d, y_test, snr_test)
print_results(f'Train-High ({HIGH_SNRS}dB) -> Test-All', hl_accs)

low_mean_hl   = np.mean([a for snr,a in zip(SNRs_to_test, hl_accs) if snr in LOW_SNRS])
low_mean_full = np.mean([a for snr,a in zip(SNRs_to_test, ablation_results['Full Model']) if snr in LOW_SNRS])
print(f"\n  Low-SNR ({LOW_SNRS}dB) mean accuracy:")
print(f"    Train-High model : {low_mean_hl:.2f}%")
print(f"    Full-range model : {low_mean_full:.2f}%")
print(f"    Gap              : {low_mean_full - low_mean_hl:+.2f}pp")
print("  Interpretation: a large gap indicates the model relies partly on "
      "SNR-specific noise statistics seen during training rather than purely "
      "modulation-invariant features; a small gap supports modulation-"
      "invariant feature learning (R3 pt.9).")
print(f"\n  ⏱  {(time.time()-t0)/60:.1f} min")
tf.keras.backend.clear_session(); gc.collect()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 5. PLOTS + EXPORT
# ══════════════════════════════════════════════════════════════════════════

# ── Ablation bar chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10,6))
names  = ['Full Model', 'w/o GASF-CNN', 'w/o Attention Gate',
          'w/o Label Smoothing', 'Lean (no redundant EMD)']
means  = [np.mean(ablation_results[n]) for n in names]
colors = ['#E84855', '#48CAE4', '#F4A261', '#9B59B6', '#6A994E']
bars = ax.bar(names, means, color=colors, alpha=0.85)
for bar, m in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, m+0.5, f'{m:.2f}%', ha='center', fontweight='bold')
ax.set_ylabel('Mean Accuracy across SNRs (%)')
ax.set_title('Ablation Study — Real Measured Results (R3 pt.8)', fontweight='bold')
ax.set_ylim(0, 105)
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'p30_ablation_study_measured.png'))
plt.show()
print("Saved: p30_ablation_study_measured.png")

# ── Ablation accuracy vs SNR ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10,6.5))
markers = ['*','^','s','D','o']
for name, mk, clr in zip(names, markers, colors):
    lw = 2.8 if name == 'Full Model' else 1.8
    ax.plot(SNRs_to_test, ablation_results[name], marker=mk, ms=10 if name=='Full Model' else 7,
            lw=lw, color=clr, label=name)
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Ablation Study — Accuracy vs SNR (measured)', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.4)
ax.set_xlim(-12,22); ax.set_ylim(20,105)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'p31_ablation_vs_snr.png'))
plt.show()
print("Saved: p31_ablation_vs_snr.png")

# ── FSK per-SNR plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9,6))
ax.plot(SNRs_to_test, fsk_accs, marker='*', ms=12, lw=2.8, color='#E84855', label='5-class overall')
ax.plot(SNRs_to_test, fsk_per_snr, marker='D', ms=8, lw=2, color='#2E4057', label='FSK class only')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Accuracy (%)')
ax.set_title('5-Class Model (incl. FSK) — Accuracy vs SNR (R4 pt.6)', fontweight='bold')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.4)
ax.set_xlim(-12,22); ax.set_ylim(0,105)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'p32_fsk_5class.png'))
plt.show()
print("Saved: p32_fsk_5class.png")

# ── Train-High/Test-Low comparison plot ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9,6))
ax.plot(SNRs_to_test, ablation_results['Full Model'], marker='*', ms=12, lw=2.8,
        color='#E84855', label='Full-range trained')
ax.plot(SNRs_to_test, hl_accs, marker='s', ms=8, lw=2, color='#48CAE4',
        label=f'Train-High only ({HIGH_SNRS}dB)')
ax.axvspan(-12, 1.5, alpha=0.1, color='gray', label='Low-SNR test region')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Accuracy (%)')
ax.set_title('Train-High/Test-Low Generalization (R3 pt.9)', fontweight='bold')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.4)
ax.set_xlim(-12,22); ax.set_ylim(0,105)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'p33_train_high_test_low.png'))
plt.show()
print("Saved: p33_train_high_test_low.png")

# ── Export everything ────────────────────────────────────────────────────
nb2_export = {
    'ablation_results': ablation_results,
    'ablation_params':  ablation_params,
    'fsk_accs': fsk_accs,
    'fsk_per_snr': fsk_per_snr,
    'hl_accs': hl_accs,
    'low_mean_hl': low_mean_hl,
    'low_mean_full': low_mean_full,
    'SNRs_to_test': SNRs_to_test,
    'HIGH_SNRS': HIGH_SNRS,
    'LOW_SNRS': LOW_SNRS,
}
with open('/kaggle/working/nb2_export.pkl', 'wb') as f:
    pickle.dump(nb2_export, f)
print("\nSaved nb2_export.pkl")

import glob
pngs = glob.glob(os.path.join(OUTPUT_DIR, '*.png'))
if pngs:
    os.system(f'zip -r /kaggle/working/all_figures_nb2.zip {OUTPUT_DIR}/')
    print(f'Zipped {len(pngs)} figures → /kaggle/working/all_figures_nb2.zip')
